In [1]:
import torch
import torch.nn as nn

class WeightedRSRPModel(nn.Module):
    def __init__(self, dim=7):
        super().__init__()
        # Konum tahmin vektörleri
        self.u_x = nn.Parameter(torch.randn(dim))
        self.u_y = nn.Parameter(torch.randn(dim))
        self.v_x = nn.Parameter(torch.randn(dim))
        self.v_y = nn.Parameter(torch.randn(dim))

        # Öğrenilecek ağırlıklar (gerçek değerleri sigmoid ile [0,1] yapılır)
        self.w_x_raw = nn.Parameter(torch.tensor(0.0))  # Başlangıç = 0 (sigmoid(0) = 0.5)
        self.w_y_raw = nn.Parameter(torch.tensor(0.0))

    def forward(self, q, x, y, p, x_m, y_m, lambda1=0.01, lambda2=0.01):
        # Sigmoid ile ağırlıkları [0,1] aralığına getir
        W_x = torch.sigmoid(self.w_x_raw)
        W_y = torch.sigmoid(self.w_y_raw)

        # Simülasyon tahminleri
        pred_x_sim = q @ self.u_x
        pred_y_sim = q @ self.u_y

        # Ölçüm tahminleri
        pred_x_meas = p @ self.v_x
        pred_y_meas = p @ self.v_y

        # Ağırlıklı loss
        loss_x = (1 - W_x) * torch.mean((pred_x_sim - x) ** 2) + \
                  W_x * torch.mean((pred_x_meas - x_m) ** 2)

        loss_y = (1 - W_y) * torch.mean((pred_y_sim - y) ** 2) + \
                  W_y * torch.mean((pred_y_meas - y_m) ** 2)

        # L2 regularizasyon
        reg = lambda1 * (self.u_x.norm() + self.u_y.norm()) + \
              lambda2 * (self.v_x.norm() + self.v_y.norm())

        total_loss = loss_x + loss_y + reg
        return total_loss, W_x.item(), W_y.item()

def train_weighted_model(model, q, x, y, p, x_m, y_m, lambda1=0.01, lambda2=0.01, lr=0.001, steps=10000):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    for step in range(steps):
        optimizer.zero_grad()
        loss, Wx, Wy = model(q, x, y, p, x_m, y_m, lambda1, lambda2)
        loss.backward()
        optimizer.step()

        if step % 100 == 0:
            print(f"[{step}] Loss: {loss.item():.4f}, Wx: {Wx:.3f}, Wy: {Wy:.3f}")


In [2]:
# Veri örnekleri
N, M, D = 10000, 200, 7

q = torch.randn(N, D)
x = torch.randn(N)
y = torch.randn(N)

p = torch.randn(M, D)
x_m = torch.randn(M)
y_m = torch.randn(M)

# Model
model = WeightedRSRPModel(dim=D)

# Eğitim
train_weighted_model(model, q, x, y, p, x_m, y_m)

[0] Loss: 17.1439, Wx: 0.500, Wy: 0.500
[100] Loss: 13.7935, Wx: 0.477, Wy: 0.466
[200] Loss: 11.2758, Wx: 0.461, Wy: 0.439
[300] Loss: 9.3412, Wx: 0.449, Wy: 0.417
[400] Loss: 7.8347, Wx: 0.440, Wy: 0.399
[500] Loss: 6.6517, Wx: 0.433, Wy: 0.383
[600] Loss: 5.7178, Wx: 0.428, Wy: 0.370
[700] Loss: 4.9780, Wx: 0.424, Wy: 0.359
[800] Loss: 4.3904, Wx: 0.421, Wy: 0.349
[900] Loss: 3.9227, Wx: 0.419, Wy: 0.341
[1000] Loss: 3.5496, Wx: 0.417, Wy: 0.333
[1100] Loss: 3.2515, Wx: 0.416, Wy: 0.327
[1200] Loss: 3.0127, Wx: 0.415, Wy: 0.321
[1300] Loss: 2.8212, Wx: 0.415, Wy: 0.316
[1400] Loss: 2.6671, Wx: 0.414, Wy: 0.311
[1500] Loss: 2.5428, Wx: 0.414, Wy: 0.307
[1600] Loss: 2.4424, Wx: 0.414, Wy: 0.304
[1700] Loss: 2.3609, Wx: 0.414, Wy: 0.301
[1800] Loss: 2.2947, Wx: 0.415, Wy: 0.298
[1900] Loss: 2.2406, Wx: 0.415, Wy: 0.295
[2000] Loss: 2.1964, Wx: 0.415, Wy: 0.293
[2100] Loss: 2.1600, Wx: 0.416, Wy: 0.291
[2200] Loss: 2.1301, Wx: 0.417, Wy: 0.289
[2300] Loss: 2.1053, Wx: 0.417, Wy: 0.288
[

In [5]:
q[0]

tensor([-0.0042, -0.9887, -0.2778, -0.6951, -0.8388,  0.4840, -0.1329])